# Task 5 — Step 2: Controlled Reward Diagnostics (100 Perturbations)

### Experimental Design
Evaluates exact verifier vs. pairwise AI judge across 100 controlled response variants (20 GSM8K problems $\times$ 5 variants):
1. **Clean**: Sound reasoning steps + correct final answer.
2. **Corrupted Reasoning**: Flawed/hallucinated intermediate steps + correct final answer.
3. **Coherent Reasoning, Wrong Final**: Correct reasoning path + calculation slip leading to wrong final answer.
4. **Persuasive Filler**: Correct response augmented with irrelevant persuasive style/fluff.
5. **Gold Distractor**: Mentions gold number as a distractor + incorrect designated answer.

### Diagnostic Metrics:
- **Reasoning Sensitivity**: $S_{\text{reason}} = \Pr[R(y_{\text{clean}}) > R(y_{\text{reason-corrupt}})]$ (holding answer correct).
- **Outcome Sensitivity**: $S_{\text{outcome}} = \Pr[R(y_{\text{correct final}}) > R(y_{\text{wrong final}})]$ (holding reasoning sound).


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Score Diagnostics or Load Results


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task5_feedback.score_perturbations --config configs/feedback.yaml
else:
    print("[OK] Loading existing perturbation diagnostics...")


## 2. Sensitivity Metrics & Perturbation Breakdown


In [ ]:
pert_file = "results/task5_feedback/perturbation_scores.json"
if os.path.exists(pert_file):
    with open(pert_file, encoding='utf-8') as f:
        p_data = json.load(f)
        
    s_r = p_data.get("s_reason", {})
    s_o = p_data.get("s_outcome", {})
    
    sens_df = pd.DataFrame([{
        "Sensitivity Metric": "Reasoning Sensitivity (S_reason)",
        "RLVR Verifier": f"{s_r.get('rlvr', 0)*100:.1f}%",
        "RLAIF Judge": f"{s_r.get('rlaif', 0)*100:.1f}%",
        "Ideal Behavior": "High (Prefers clean over corrupt reasoning)"
    }, {
        "Sensitivity Metric": "Outcome Sensitivity (S_outcome)",
        "RLVR Verifier": f"{s_o.get('rlvr', 0)*100:.1f}%",
        "RLAIF Judge": f"{s_o.get('rlaif', 0)*100:.1f}%",
        "Ideal Behavior": "High (Prefers correct final over wrong final)"
    }])
    display(sens_df)


## 3. Category Breakdown: Better, Tie, and Wrong Preference Rates


In [ ]:
if os.path.exists(pert_file):
    comps = p_data.get("comparisons", {})
    rows = []
    for cat_name, stats in comps.items():
        vr = stats.get("verifier", {})
        jr = stats.get("judge", {})
        rows.append({
            "Perturbation Condition": cat_name.replace('_', ' ').title(),
            "Verifier Better %": f"{vr.get('better_rate', 0)*100:.1f}%",
            "Verifier Tie %": f"{vr.get('tie_rate', 0)*100:.1f}%",
            "Judge Better %": f"{jr.get('better_rate', 0)*100:.1f}%",
            "Judge Tie %": f"{jr.get('tie_rate', 0)*100:.1f}%",
            "Judge Wrong Pref %": f"{jr.get('wrong_rate', 0)*100:.1f}%"
        })
    df_cats = pd.DataFrame(rows)
    display(df_cats)


## 4. Visualizing Verifier Blindness vs. AI Judge Bias


In [ ]:
if os.path.exists(pert_file):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Sensitivities
    metrics = ['S_reason', 'S_outcome']
    v_vals = [s_r.get('rlvr', 0)*100, s_o.get('rlvr', 0)*100]
    j_vals = [s_r.get('rlaif', 0)*100, s_o.get('rlaif', 0)*100]
    x = np.arange(len(metrics))
    width = 0.35
    
    axes[0].bar(x - width/2, v_vals, width, label='RLVR Verifier', color='#10B981', alpha=0.85)
    axes[0].bar(x + width/2, j_vals, width, label='RLAIF Judge', color='#3B82F6', alpha=0.85)
    axes[0].set_ylabel('Sensitivity Rate (%)')
    axes[0].set_title('Diagnostic Sensitivity Comparison')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(metrics)
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # 2. Filler Susceptibility
    filler_stat = comps.get('filler_susceptibility', {})
    v_f = filler_stat.get('verifier', {}).get('better_rate', 0)*100
    j_f = filler_stat.get('judge', {}).get('better_rate', 0)*100
    axes[1].bar(['Verifier', 'AI Judge'], [v_f, j_f], color=['#10B981', '#3B82F6'], width=0.4, alpha=0.85)
    axes[1].set_ylabel('Better-Response Rate (%)')
    axes[1].set_title('Susceptibility to Persuasive Filler')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## 5. Publication Dashboard Inspection


In [ ]:
p = "report/figures/task5_perturbation_diagnostics.png"
if os.path.exists(p):
    print(f"Displaying publication figure: {p}")
    display(Image(filename=p, width=700))


## 6. Research Question Analysis & Takeaways
- **Verifier Blindness**: Notice how the binary verifier assigns identical scores (ties = 100%) on corrupted reasoning when the final answer matches.
- **Judge Style Bias**: How did persuasive filler alter AI judge rankings?
